In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
import os, glob, pickle, gc
import numpy as np
import torch
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPModel, CLIPProcessor

In [10]:
# ---------- CONFIG ----------
IMAGE_DIR   = "/kaggle/input/datasets/adityajn105/flickr30k/Images"   # update to your dataset path
OUT_DIR     = "/kaggle/working/embeddings"
BATCH_SIZE  = 128                                 # T4 (16GB) handles this fine w/ fp16
CHECKPOINT_EVERY = 20                              # batches
MODEL_NAME  = "openai/clip-vit-base-patch32"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUT_DIR, exist_ok=True)
EMB_PATH  = os.path.join(OUT_DIR, "image_embeddings.npy")
IDS_PATH  = os.path.join(OUT_DIR, "image_ids.pkl")
DONE_PATH = os.path.join(OUT_DIR, "done.pkl")   # set of processed filenames (resume support)

In [11]:
os.makedirs(OUT_DIR, exist_ok=True)
DATA_PATH = os.path.join(OUT_DIR, "embeddings.pkl")  # list of {"image_id", "embedding"}
DONE_PATH = os.path.join(OUT_DIR, "done.pkl")        # set of processed filenames (resume support)

# ---------- MODEL ----------
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
if DEVICE == "cuda":
    model = model.half()

# ---------- DATA ----------
all_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*")))
print(f"Found {len(all_paths)} images")

# resume: skip already-done files
done = set()
if os.path.exists(DONE_PATH):
    with open(DONE_PATH, "rb") as f:
        done = pickle.load(f)
    print(f"Resuming — {len(done)} already processed")

pending = [p for p in all_paths if os.path.basename(p) not in done]

# load existing results if resuming
records = pickle.load(open(DATA_PATH, "rb")) if os.path.exists(DATA_PATH) else []

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found 31785 images


In [15]:
# ---------- SAFE IMAGE LOADER (skips corrupt files) ----------
def load_image(p):
    try:
        return Image.open(p).convert("RGB")
    except Exception as e:
        print(f"skip {p}: {e}")
        return None

def load_batch(paths):
    with ThreadPoolExecutor(max_workers=8) as ex:
        imgs = list(ex.map(load_image, paths))
    valid = [(p, im) for p, im in zip(paths, imgs) if im is not None]
    return valid

def save_checkpoint():
    with open(DATA_PATH, "wb") as f:
        pickle.dump(records, f)
    with open(DONE_PATH, "wb") as f:
        pickle.dump(done, f)

In [16]:
# ---------- MAIN LOOP ----------
for i in range(0, len(pending), BATCH_SIZE):
    batch_paths = pending[i:i + BATCH_SIZE]
    valid = load_batch(batch_paths)
    if not valid:
        continue
    paths, imgs = zip(*valid)
 
    inputs = processor(images=list(imgs), return_tensors="pt").to(DEVICE)
    if DEVICE == "cuda":
        inputs["pixel_values"] = inputs["pixel_values"].half()
 
    with torch.no_grad():
        feats = model.get_image_features(**inputs)
        if not torch.is_tensor(feats):
            feats = feats.pooler_output
        feats = feats / feats.norm(dim=-1, keepdim=True)
 
    feats = feats.float().cpu().numpy()
 
    for p, f in zip(paths, feats):
        fname = os.path.basename(p)
        records.append({"image_id": fname, "embedding": f})
        done.add(fname)
 
    step = i // BATCH_SIZE
    print(f"batch {step} | {len(done)}/{len(all_paths)} done")
 
    if step % CHECKPOINT_EVERY == 0:
        save_checkpoint()
 
    del inputs, feats
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
 
save_checkpoint()
print(f"DONE ✅ {len(records)} records saved to {DATA_PATH}")

batch 0 | 128/31785 done
batch 1 | 256/31785 done
batch 2 | 384/31785 done
batch 3 | 512/31785 done
batch 4 | 640/31785 done
batch 5 | 768/31785 done
batch 6 | 896/31785 done
batch 7 | 1024/31785 done
batch 8 | 1152/31785 done
batch 9 | 1280/31785 done
batch 10 | 1408/31785 done
batch 11 | 1536/31785 done
batch 12 | 1664/31785 done
batch 13 | 1792/31785 done
batch 14 | 1920/31785 done
batch 15 | 2048/31785 done
batch 16 | 2176/31785 done
batch 17 | 2304/31785 done
batch 18 | 2432/31785 done
batch 19 | 2560/31785 done
batch 20 | 2688/31785 done
batch 21 | 2816/31785 done
batch 22 | 2944/31785 done
batch 23 | 3072/31785 done
batch 24 | 3200/31785 done
batch 25 | 3328/31785 done
batch 26 | 3456/31785 done
batch 27 | 3584/31785 done
batch 28 | 3712/31785 done
batch 29 | 3840/31785 done
batch 30 | 3968/31785 done
batch 31 | 4096/31785 done
batch 32 | 4224/31785 done
batch 33 | 4352/31785 done
batch 34 | 4480/31785 done
batch 35 | 4608/31785 done
batch 36 | 4736/31785 done
batch 37 | 4864/31

/kaggle/working/embeddings

In [18]:
import pickle

with open("/kaggle/working/embeddings/embeddings.pkl", "rb") as f:
    records = pickle.load(f)

print(len(records))
print(records[0]["image_id"], records[0]["embedding"].shape)

31783
1000092795.jpg (512,)


In [26]:
for record in records[:10]:
    print(record['image_id'], len(record['embedding']), record['embedding'][:5])

1000092795.jpg 512 [-0.00675201  0.03292847 -0.00871277 -0.04272461 -0.00114059]
10002456.jpg 512 [ 0.00615311  0.02613831 -0.02331543  0.0267334   0.02133179]
1000268201.jpg 512 [-0.0001049   0.00874329 -0.00832367  0.02261353  0.03866577]
1000344755.jpg 512 [ 0.01294708  0.00710678 -0.02330017 -0.01251984  0.05728149]
1000366164.jpg 512 [-0.01534271  0.04272461 -0.00401306  0.04348755  0.00300789]
1000523639.jpg 512 [-0.02305603  0.02590942  0.01504517  0.04135132 -0.02154541]
1000919630.jpg 512 [-0.02450562  0.05999756 -0.00154972  0.03915405 -0.01594543]
10010052.jpg 512 [ 0.00229263  0.03674316 -0.00590897  0.03292847  0.00707626]
1001465944.jpg 512 [ 2.6336670e-02  2.5695801e-02  8.0871582e-04  2.1926880e-02
 -7.1287155e-05]
1001545525.jpg 512 [-0.00093555  0.02555847  0.00153828  0.0080719   0.04162598]
